# PRAHARI — model training

Fits the two models the screening engine uses, and shows the evidence behind the
thresholds they run at.

**Runs standalone.** It generates its own data and clones nothing, so it works on a
fresh Colab runtime with no repository checkout.

---

### What this notebook is, and is not

It is evidence that the models are trained rather than asserted, and that the 0.82
cosine threshold was chosen from a distribution rather than picked.

It is **not** a measurement of real-world accuracy. Every anomaly it measures against
was planted by this code. A high recall shows the detectors are wired to the patterns
they were built for — nothing more. Validating against live MPLADS records is a
different exercise, and would need data nobody publishes.


In [ ]:
!pip -q install scikit-learn sentence-transformers matplotlib joblib pandas


In [ ]:
import math, random
from datetime import date, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Deterministic under a seed, exactly as the backend generator is, so every
# figure below is reproducible.
SEED = 42
rng = random.Random(SEED)
np.random.seed(SEED)

REFERENCE_DATE = date(2026, 8, 31)
print('reference date:', REFERENCE_DATE)


## 1. Generate the corpus

The same shape the backend seeds: works priced log-normally around a Schedule of
Rates benchmark, with a labelled minority carrying planted anomalies.

The benchmark escalates 6.2% a year, and each work is compared against the rate for
**its own year**. That is the inflation defence: when costs and rates rise together
the ratio does not move, so no finding fires. Section 3 checks it rather than
asserting it.


In [ ]:
WORK_TYPES = {
    'ROAD_CC': 1_890_000, 'ROAD_BT': 1_395_000, 'COMMUNITY_HALL': 832_500,
    'SCHOOL_BUILDING': 427_500, 'WATER_TANK': 517_500, 'BOREWELL': 216_000,
    'STREET_LIGHTING': 144_000, 'DRAINAGE': 184_500, 'TOILET_BLOCK': 279_000,
    'LIBRARY': 630_000, 'BUS_SHELTER': 126_000, 'CREMATORIUM_SHED': 243_000,
}
TERRAIN = {'PLAIN': 1.00, 'COASTAL': 1.10, 'URBAN': 1.18, 'HILLY': 1.28, 'REMOTE': 1.42}
ESCALATION, BASE_YEAR = 0.062, 2023

def sor_rate(work_type, year, terrain):
    base = WORK_TYPES[work_type] * ((1 + ESCALATION) ** (year - BASE_YEAR))
    return round(base * TERRAIN[terrain], 2)

ACTIONS = ['Construction of', 'Providing and laying', 'Development of', 'Erection of']
DETAIL = {
    'ROAD_CC': ['cement concrete road with side drains', 'CC road including earthwork'],
    'ROAD_BT': ['bituminous surfacing over existing WBM road', 'black-topped approach road'],
    'COMMUNITY_HALL': ['community hall with RCC roof', 'multipurpose community building'],
    'SCHOOL_BUILDING': ['two additional classrooms with verandah', 'school block with RCC slab'],
    'WATER_TANK': ['RCC overhead water storage tank', 'elevated storage tank with pipeline'],
    'BOREWELL': ['borewell with submersible pump', 'tubewell with concrete platform'],
    'STREET_LIGHTING': ['solar LED street lights with poles', 'street lighting with GI poles'],
    'DRAINAGE': ['covered pucca drainage line with slabs', 'RCC drainage channel'],
    'TOILET_BLOCK': ['community toilet block with septic tank', 'public sanitation block'],
    'LIBRARY': ['village reading room and library', 'library building with shelving'],
    'BUS_SHELTER': ['passenger bus shelter with seating', 'bus waiting shed with roofing'],
    'CREMATORIUM_SHED': ['covered crematorium shed', 'cremation ground shed with platform'],
}
PLACES = ['Bhuwana','Debari','Kherwara','Mavli','Badgaon','Sarada','Gogunda','Kotra']

def describe(work_type, place):
    return f'{rng.choice(ACTIONS)} {rng.choice(DETAIL[work_type])} at {place} village.'


In [ ]:
N_WORKS = 3000
PER_ANOMALY = 60

rows = []
for i in range(N_WORKS):
    terrain = rng.choice(list(TERRAIN))
    work_type = rng.choice(list(WORK_TYPES))
    place = rng.choice(PLACES)
    recommended = REFERENCE_DATE - timedelta(days=rng.randint(20, 1080))
    year = min(recommended.year, 2026)
    benchmark = sor_rate(work_type, year, terrain)
    # sigma keeps about 95% inside +/-25%, so an ordinary work does not trip
    # the +25% cost threshold by itself.
    cost = round(benchmark * rng.lognormvariate(0, 0.11), 2)
    rows.append(dict(
        work_id=f'W-{i:05d}', work_type=work_type, terrain=terrain,
        year=year, benchmark=benchmark, estimated_cost=cost,
        description=describe(work_type, place),
        latitude=round(24.0 + rng.uniform(-0.4, 0.4), 6),
        longitude=round(73.7 + rng.uniform(-0.4, 0.4), 6),
        recommended=recommended, agency=f'AG-{rng.randint(1, 24):02d}',
        planted=None,
    ))

df = pd.DataFrame(rows)
print(f'{len(df):,} works generated')
df[['work_id','work_type','terrain','year','benchmark','estimated_cost']].head()


### Plant the labelled anomalies

Two matter for the models trained here: inflated costs, and duplicate pairs. The
duplicate pairs are what section 4 uses to choose the similarity threshold.


In [ ]:
candidates = df.index[df.planted.isna()].tolist()
rng.shuffle(candidates)

# COST_INFLATION - 35% to 120% above the same-year benchmark.
for idx in candidates[:PER_ANOMALY]:
    df.at[idx, 'estimated_cost'] = round(df.at[idx, 'benchmark'] * rng.uniform(1.35, 2.20), 2)
    df.at[idx, 'planted'] = 'COST_INFLATION'

# DUPLICATE_WORK - a near-copy within roughly 200 m and 180 days. Only the later
# work is labelled; it is the one a screener should catch.
pool = candidates[PER_ANOMALY:]
duplicate_pairs = []
for k in range(0, PER_ANOMALY * 2, 2):
    a, b = pool[k], pool[k + 1]
    df.at[b, 'work_type'] = df.at[a, 'work_type']
    df.at[b, 'description'] = (df.at[a, 'description']
        .replace('Construction of', 'Providing and laying')
        .replace('Development of', 'Construction of'))
    df.at[b, 'latitude'] = round(df.at[a, 'latitude'] + rng.uniform(-0.0015, 0.0015), 6)
    df.at[b, 'longitude'] = round(df.at[a, 'longitude'] + rng.uniform(-0.0015, 0.0015), 6)
    df.at[b, 'recommended'] = df.at[a, 'recommended'] + timedelta(days=rng.randint(10, 175))
    df.at[b, 'planted'] = 'DUPLICATE_WORK'
    duplicate_pairs.append((df.at[a, 'work_id'], df.at[b, 'work_id']))

print(df.planted.value_counts(dropna=False))
print(f'{len(duplicate_pairs)} duplicate pairs planted')


## 2. Feature engineering

Every feature is a **ratio or a count**, never a rupee amount. Raw cost would let the
model learn that expensive work types are unusual, which is a fact about road-building
rather than a finding.

Age is deliberately **not** a feature. Only works awaiting sanction reach Stage 1 and
those are all recent, so a recency feature teaches the model that new works are
strange. In an earlier version of the engine it flagged 39% of proposals for no
better reason than that.


In [ ]:
df['cost_ratio'] = df.estimated_cost / df.benchmark

# Peer statistics per (work type, terrain). Median and MAD rather than mean and
# standard deviation, because the thing being detected is an outlier and outliers
# drag a mean towards themselves.
peer = df.groupby(['work_type','terrain']).cost_ratio.agg(['median','count'])
peer['mad'] = df.groupby(['work_type','terrain']).cost_ratio.apply(
    lambda s: (s - s.median()).abs().median())

def modified_z(row):
    stats = peer.loc[(row.work_type, row.terrain)]
    if stats['mad'] <= 0 or stats['count'] < 8:
        return 0.0
    # 0.6745 makes MAD a consistent estimator of the standard deviation for
    # normally distributed data.
    return 0.6745 * (row.cost_ratio - stats['median']) / stats['mad']

df['cost_peer_z'] = df.apply(modified_z, axis=1)

coords = df[['latitude','longitude']].to_numpy()
def nearby(i):
    d = np.sqrt(((coords - coords[i]) ** 2).sum(axis=1)) * 111_000
    return int((d <= 500).sum() - 1)
df['works_in_500m'] = [nearby(i) for i in range(len(df))]
df['agency_load'] = df.groupby('agency').work_id.transform('count')

FEATURES = ['cost_ratio', 'cost_peer_z', 'works_in_500m', 'agency_load']
X = pd.get_dummies(df[FEATURES + ['work_type','terrain']], columns=['work_type','terrain'])
print('feature matrix:', X.shape)
X.head()


## 3. The inflation defence, checked

The claim: comparing a cost against the Schedule of Rates **for its own year** makes
the comparison inflation-neutral, so a general price rise produces no findings.

If it holds, the nominal benchmark climbs across the period while the cost ratio
stays flat. If it does not, this plot shows it immediately.


In [ ]:
clean = df[df.planted.isna()]
by_year = clean.groupby('year').agg(
    median_benchmark=('benchmark','median'),
    median_ratio=('cost_ratio','median'),
    p95_ratio=('cost_ratio', lambda s: s.quantile(0.95)),
    n=('work_id','count'),
)
display(by_year.round(3))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))
ax1.bar(by_year.index.astype(str), by_year.median_benchmark, color='#16457e')
ax1.set_title('Median benchmark (nominal rupees)', fontsize=10)
ax1.set_ylabel('Rs')

ax2.plot(by_year.index.astype(str), by_year.median_ratio, marker='o',
         color='#16457e', label='median')
ax2.plot(by_year.index.astype(str), by_year.p95_ratio, marker='o', linestyle='--',
         color='#8a939b', label='p95')
ax2.axhline(1.25, color='#ae1414', linestyle=':', label='flag line (+25%)')
ax2.set_ylim(0.8, 1.45)
ax2.set_title('Cost / same-year benchmark', fontsize=10)
ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

spread = by_year.median_ratio.max() - by_year.median_ratio.min()
growth = by_year.median_benchmark.iloc[-1] / by_year.median_benchmark.iloc[0] - 1
print(f'nominal benchmark growth across the period : {growth:.1%}')
print(f'spread in median cost ratio                : {spread:.3f}')
print('inflation neutral:', 'YES' if spread < 0.02 else 'NO')


## 4. Choosing the similarity threshold

The duplicate module runs at cosine >= 0.82. That number should come from a
distribution, not from taste.

Below: the similarity of **known duplicate pairs** against **random unrelated pairs**,
using the same `all-MiniLM-L6-v2` embeddings the backend uses. A defensible
threshold sits in the gap between the two.


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
texts = df.set_index('work_id').description.to_dict()
embeddings = model.encode(list(texts.values()), batch_size=64,
                          normalize_embeddings=True, show_progress_bar=True)
vec = dict(zip(texts.keys(), embeddings))
print('embedded', len(vec), 'descriptions')


In [ ]:
def cosine(a, b):
    return float(vec[a] @ vec[b])

positive = [cosine(a, b) for a, b in duplicate_pairs]

ids = df.work_id.tolist()
in_pairs = {w for pair in duplicate_pairs for w in pair}
negative = []
while len(negative) < 3000:
    a, b = rng.sample(ids, 2)
    if a in in_pairs and b in in_pairs:
        continue
    negative.append(cosine(a, b))

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(negative, bins=60, alpha=0.75, color='#8a939b',
        label=f'unrelated pairs (n={len(negative)})')
ax.hist(positive, bins=30, alpha=0.85, color='#16457e',
        label=f'known duplicates (n={len(positive)})')
ax.axvline(0.82, color='#ae1414', linestyle='--', label='threshold 0.82')
ax.set_xlabel('cosine similarity'); ax.set_ylabel('pairs'); ax.legend(fontsize=8)
ax.set_title('Description similarity: duplicates vs unrelated works', fontsize=10)
plt.tight_layout(); plt.show()

pos, neg = np.array(positive), np.array(negative)
print(f'duplicates : median {np.median(pos):.3f}   5th pct {np.percentile(pos, 5):.3f}')
print(f'unrelated  : median {np.median(neg):.3f}  95th pct {np.percentile(neg, 95):.3f}')
print()
for t in [0.70, 0.75, 0.80, 0.82, 0.85, 0.90]:
    print(f'  threshold {t:.2f}   recall {(pos >= t).mean():6.1%}'
          f'   unrelated pairs above it {(neg >= t).mean():6.2%}')


**Reading that table.** Similarity alone is a weak signal — plenty of unrelated works
are described almost identically, because most road descriptions read alike. That is
exactly why the engine requires similarity **and** proximity within 500 m **and** a
365-day window before it raises anything. The threshold is one of three conditions,
not the whole test.


## 5. Fit the Isolation Forest

Unsupervised, because no labelled dataset of MPLADS irregularities exists. The model
learns what the ordinary population looks like and reports how far a proposal sits
from it. It never sees the `planted` column.


In [ ]:
from sklearn.ensemble import IsolationForest

forest = IsolationForest(n_estimators=200, contamination=0.10, random_state=42, n_jobs=-1)
forest.fit(X.astype(float))

raw = forest.decision_function(X.astype(float))
training_scores = np.sort(raw)

def percentile_score(value):
    """0-100, where 78 means stranger than 78% of the corpus.

    Rank rather than min-max. Scaled linearly, decision_function output is so
    tightly clustered that almost nothing crosses a threshold and the published
    number corresponds to nothing a reader could name.
    """
    rank = np.searchsorted(training_scores, value, side='left')
    return 100.0 - (rank / len(training_scores)) * 100.0

df['isolation_score'] = [percentile_score(v) for v in raw]

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.hist(df[df.planted.isna()].isolation_score, bins=50, alpha=0.75,
        color='#8a939b', label='no planted anomaly')
ax.hist(df[df.planted.notna()].isolation_score, bins=50, alpha=0.85,
        color='#16457e', label='planted anomaly')
ax.axvline(90, color='#ae1414', linestyle='--', label='flag line (90)')
ax.set_xlabel('isolation score (percentile)'); ax.set_ylabel('works')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## 6. Method sensitivity

Recall per planted pattern, matching what the CAG backtest screen reports. Both are
computed the same way, so the notebook and the dashboard cannot disagree.

**These numbers measure the method against anomalies planted here.** They are
evidence the detectors work as built, not evidence of real-world accuracy.


In [ ]:
cost_flagged = (df.cost_ratio > 1.25) | (df.cost_peer_z > 3.5)

# The duplicate rule as the engine applies it: similarity AND proximity AND window.
lookup = df.set_index('work_id')
def is_duplicate(work_id):
    row = lookup.loc[work_id]
    same_type = df[(df.work_type == row.work_type) & (df.work_id != work_id)]
    for _, other in same_type.iterrows():
        metres = math.hypot(row.latitude - other.latitude,
                            row.longitude - other.longitude) * 111_000
        if metres > 500 or abs((row.recommended - other.recommended).days) > 365:
            continue
        if cosine(work_id, other.work_id) >= 0.82:
            return True
    return False

results = []
for pattern in ['COST_INFLATION', 'DUPLICATE_WORK']:
    members = df[df.planted == pattern]
    caught = (int(cost_flagged[members.index].sum()) if pattern == 'COST_INFLATION'
              else sum(is_duplicate(w) for w in members.work_id))
    results.append(dict(pattern=pattern, planted=len(members), recalled=caught,
                        recall=caught / len(members)))

clean_flagged = int(cost_flagged[df.planted.isna()].sum())
n_clean = int(df.planted.isna().sum())

display(pd.DataFrame(results).style.format({'recall': '{:.1%}'}))
print(f'works with no planted anomaly flagged on cost: '
      f'{clean_flagged}/{n_clean} = {clean_flagged / n_clean:.2%}')


## 7. Export the model

The backend fits this model in-process when it scores, so the artefact is not a hard
dependency. It is evidence that training happened, and something a reviewer can load
and inspect.


In [ ]:
import joblib

artefact = {
    'model': forest,
    'feature_names': list(X.columns),
    'training_scores': training_scores,
    'seed': SEED,
    'n_training_works': len(df),
    'contamination': 0.10,
    'flag_threshold_percentile': 90,
    'similarity_threshold': 0.82,
    'note': 'Fitted on synthetic MPLADS-shaped data. Not validated against live records.',
}
joblib.dump(artefact, 'isolation_forest.joblib')
print('written: isolation_forest.joblib')
print()
print('Download it from the file browser in the left sidebar, then place it at:')
print('  backend/app/engine/artifacts/isolation_forest.joblib')


---

## What to take from this

| Section | What it establishes |
|---|---|
| 3 | The inflation defence holds: nominal rates rise, the cost ratio does not move |
| 4 | The 0.82 similarity threshold sits in a real gap between duplicate and unrelated pairs — and similarity alone is too weak to use by itself |
| 5 | The Isolation Forest separates planted anomalies from ordinary works, and the published score is a percentile a reader can name |
| 6 | Recall per pattern, computed the way the dashboard computes it |

And the limit, stated once more because it is the thing most easily overclaimed:
every anomaly measured here was planted by this notebook. Nothing above is a
measurement of how the engine would perform on live MPLADS records.
